In [78]:
import confnotebook

In [79]:
from pathlib import Path

source = Path("../examples/RPA-6542/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 18470938
[1] 18470982
[2] 18548791
[3] 18561292
[4] 18561867
[5] 18639563
[6] 18660877
[7] 18667876
[8] 18667914
[9] 18668755
[10] 18674893
[11] 18687424
[12] 18690095
[13] 18690959
[14] 18692621
[15] 18699963
[16] 7-1
[17] [Untitled]_23-48


In [80]:
IDX_FILE = 9

In [81]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
output_dir = f"../examples/output/{file.stem}"

debug_image_observer = DebugImageObserver(output_dir=output_dir)

In [82]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline(debug_image=debug_image_observer)

document = pipeline.build(file.read_bytes())

Creating model: ('PP-OCRv5_server_det', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/PP-OCRv5_server_det')
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/cyrillic_PP-OCRv5_mobile_rec')
2026-07-18 09:23:01.633 | INFO     | vision_core.pipelines.build_document:build:105 - Обработка страницы 0 с dpi 200...
2026-07-18 09:23:01.647 | INFO     | vision_core.pipelines.build_document:_process_page:187 - Коррекция ориентации и наклона...
2026-07-18 09:23:01.654 | DEBUG    | vision_core.preprocessor.image_orientation:process:47 - Ориентация страницы: 0° с точностью 0.9279
2026-07-18 09:23:01.670 | DEBUG    | vision_core.preprocessor.image_orientation:compute_deskew_angle:119 - Углы наклона страницы: 0.0004°
2026-07-18 09:23:01.728 | DEBUG    | vision_core.debug_image_observer:on_debug_image:56 - Debug image saved: 3_aligned -> ../examples/output/18668755/3_aligned/page_000.png
2026-07-18 09:23:01.729 | INFO     | visi

In [83]:
from app.infrastructure.services.structured_data_extractor import ReconciliationActExtractor

extractor = ReconciliationActExtractor()
data = await extractor.extract(document)

2026-07-18 09:23:03.375 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_text:70 - summary_text: 691 символов из 2 страниц
2026-07-18 09:23:03.376 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_cell_texts:93 - summary_cell_texts: ['По данным ООО "КЛАПЕРОН",руб.', 'По данным ООО "РУССКИЙ РАДИАТОР", руб.']
2026-07-18 09:23:03.376 | DEBUG    | extractor.process:extract:19 - Текст до нормализации: Акт сверки взаимных расчетов за период: Январь 2025 г. - Март 2026 г. между О0О "КЛАПЕРОН" (ИНН 9701222720) и ООО "РУССКИЙ РАДИАТОР" (ИНН 1006013150) Мы, нижеподписавшиеся, Генеральный Директор ООО "КЛАПЕРОН" Баринова Ольга Владимировна, с одной стороны, и Генеральный Директор ООО "РуСскИй РАдИАТОР" Колпаков Никита Сергеевич, с другой стороны, составили настоящий акт сверки в том, что состояние взаимных расчетов по данным учета следующее: на 31.03.2026 задолженность в пользу О00 "КЛАПЕРОН" 115 800,00 руб. (Сто пятнадцать тысяч восемьсот 

In [84]:
print(data.debit)

[LedgerEntry(record='САЛЬДО НАЧАЛЬНОЕ', value=4935600.0, date=None, row_reference=RowReference(id_table='0', id_row='2', id_col=2, buyer_col=6)), LedgerEntry(record='ОПЛАТА (17 ОТ 13.01.2025)', value=0.0, date='13.01.2025', row_reference=RowReference(id_table='0', id_row='3', id_col=2, buyer_col=6)), LedgerEntry(record='ОПЛАТА (247 ОТ 27.01.2025)', value=0.0, date='27.01.2025', row_reference=RowReference(id_table='0', id_row='4', id_col=2, buyer_col=6)), LedgerEntry(record='ОПЛАТА (315 ОТ 03.02.2025)', value=0.0, date='03.02.2025', row_reference=RowReference(id_table='0', id_row='5', id_col=2, buyer_col=6)), LedgerEntry(record='ПРОДАЖА (4 ОТ 20.06.2025)', value=2508000.0, date='20.06.2025', row_reference=RowReference(id_table='0', id_row='6', id_col=2, buyer_col=6)), LedgerEntry(record='ОПЛАТА (1728 ОТ 09.07.2025)', value=0.0, date='09.07.2025', row_reference=RowReference(id_table='0', id_row='7', id_col=2, buyer_col=6)), LedgerEntry(record='ПРОДАЖА (6 ОТ 01.08.2025)', value=542730.0, 

In [85]:
from app.application.dto.fill_reconciliation_act import FillReconciliationActCommand
from app.domain.entities.process import ProcessState
from app.infrastructure.services.pdf_filler import DocumentPdfFiller

process_state = ProcessState(
    process_id="notebook-test",
    source_pdf=files[IDX_FILE].read_bytes(),
    document_payload=document,
)

comments = f"""
            По данным покупателя {data.buyer}
            По данным продавца {data.seller}
            В период: {data.period.start} - {data.period.end}
            """

# используем значения продавца для заполнения колонок покупателя
command = FillReconciliationActCommand(
    process_id="notebook-test",
    comments=comments,
    debit=data.debit,
    credit=data.credit,
)

filler = DocumentPdfFiller()
filled_pdf = await filler.fill(process_state, command)

2026-07-18 09:23:03.495 | INFO     | app.infrastructure.services.pdf_fill.render:resolve_font_file:221 - найден шрифт: /mnt/data/projects/rusal_recon_srv/repo/recon_vision/assets/fonts/LiberationSerif-Regular.ttf
2026-07-18 09:23:03.517 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=0 dpi=200 source=2339x1654 aligned=2339x1654 canvas=2339x1654
2026-07-18 09:23:03.532 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=1 dpi=200 source=2339x1654 aligned=2339x1654 canvas=2339x1654
2026-07-18 09:23:03.541 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R2:C6 значение=4935600.0
2026-07-18 09:23:03.541 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_font:229 - загружаем шрифт из /mnt/data/projects/rusal_recon_srv/repo/recon_vision/assets/fonts/LiberationSerif-Regular.ttf для размера 22
2026-07-18 09:23:03.542 | DEBUG    | app.infrastructure.services.pdf_filler

In [86]:
out_path = f"../examples/output/{files[IDX_FILE].stem}_filled.pdf"
Path(out_path).write_bytes(filled_pdf)
print(out_path)

../examples/output/18668755_filled.pdf
